# Piece-Square Training Notebook

End-to-end pipeline:
1. Load feature shards generated by `analysis/game_feature_extract.py`.
2. Fit a regularized linear model to predict game outcomes from piece-square occupancy.
3. Evaluate on a validation split.
4. Export learned modifiers to `data/location_modifiers.json` and (optionally) rewrite `Chess/LocationModifer.pm`.

## Prerequisites
- Create `.npz` shards with `python analysis/game_feature_extract.py lichess.pgn.zst --output-dir data/processed`.
- Install dependencies:
  ```bash
  python -m pip install numpy pandas scikit-learn matplotlib
  ```

In [ ]:
import json
import math
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    r2_score,
)


In [ ]:
REPO_ROOT = Path.cwd()
SHARD_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_JSON = REPO_ROOT / "data" / "location_modifiers.json"
UPDATE_SCRIPT = REPO_ROOT / "script" / "update_location_modifiers.pl"
PIECES = [
    "PAWN",
    "KNIGHT",
    "BISHOP",
    "ROOK",
    "QUEEN",
    "KING",
    "OPP_PAWN",
    "OPP_KNIGHT",
    "OPP_BISHOP",
    "OPP_ROOK",
    "OPP_QUEEN",
    "OPP_KING",
]
SQUARES = [f"{file}{rank}" for file in "abcdefgh" for rank in range(1, 9)]
PHASES = ["opening", "middlegame", "endgame"]
FEATURES_PER_PIECE = 64
PHASE_FEATURES = len(PHASES)
TOTAL_PIECE_FEATURES = len(PIECES) * FEATURES_PER_PIECE
FEATURE_DIM = TOTAL_PIECE_FEATURES + PHASE_FEATURES
print(f"Expecting feature dimension {FEATURE_DIM}")

In [ ]:
def load_shards(shard_dir: Path, max_shards: int | None = None):
    arrays = []
    labels = []
    weights = []
    shard_paths = sorted(shard_dir.glob("shard_*.npz"))
    if max_shards:
        shard_paths = shard_paths[:max_shards]
    if not shard_paths:
        raise FileNotFoundError(f"No shards found in {shard_dir}")

    for path in shard_paths:
        data = np.load(path)
        arrays.append(data["features"])  # shape: (N, 12*64 + 3)
        labels.append(data["labels"])
        weights.append(data["weights"])
        print(f"Loaded {path.name}: {data['features'].shape[0]} samples")

    X = np.concatenate(arrays, axis=0)
    y = np.concatenate(labels, axis=0)
    w = np.concatenate(weights, axis=0)
    assert X.shape[1] == FEATURE_DIM
    print(f"Total samples: {X.shape[0]}")
    return X, y, w

In [ ]:
# Load a manageable subset for experimentation by setting max_shards.
X, y, sample_weights = load_shards(SHARD_DIR, max_shards=None)

# Deterministic train/validation split (80/20) based on index.
indices = np.arange(X.shape[0])
split = int(0.8 * len(indices))
train_X, val_X = X[:split], X[split:]
train_y, val_y = y[:split], y[split:]
train_w, val_w = sample_weights[:split], sample_weights[split:]
print(f"Train: {train_X.shape[0]} rows, Val: {val_X.shape[0]} rows")

In [ ]:
alpha = 5.0  # tweak for regularization strength
model = Ridge(alpha=alpha, fit_intercept=False)
model.fit(train_X, train_y, sample_weight=train_w)
train_pred = model.predict(train_X)
val_pred = model.predict(val_X)
train_r2 = r2_score(train_y, train_pred, sample_weight=train_w)
val_r2 = r2_score(val_y, val_pred, sample_weight=val_w)
print(f"Train R^2: {train_r2:.4f} | Val R^2: {val_r2:.4f}")

In [ ]:
# Classification-oriented metrics derived from regression outputs

def discretize_outcomes(values, threshold=0.333):
    classes = np.zeros_like(values)
    classes[:] = 0
    classes[values > threshold] = 1
    classes[values < -threshold] = -1
    return classes

train_y_classes = discretize_outcomes(train_y)
val_y_classes = discretize_outcomes(val_y)
train_pred_classes = discretize_outcomes(train_pred)
val_pred_classes = discretize_outcomes(val_pred)

labels = [1, 0, -1]
label_names = {1: "Win", 0: "Draw", -1: "Loss"}

accuracy = accuracy_score(val_y_classes, val_pred_classes)
precision, recall, f1, support = precision_recall_fscore_support(
    val_y_classes, val_pred_classes, labels=labels, zero_division=0
)

metrics_table = [
    {
        "Class": label_names[label],
        "Precision": round(float(p), 3),
        "Recall": round(float(r), 3),
        "F1": round(float(f), 3),
        "Support": int(s),
    }
    for label, p, r, f, s in zip(labels, precision, recall, f1, support)
]

metrics_table.append(
    {
        "Class": "Macro Avg",
        "Precision": round(float(precision.mean()), 3),
        "Recall": round(float(recall.mean()), 3),
        "F1": round(float(f1.mean()), 3),
        "Support": int(support.sum()),
    }
)
metrics_table.append(
    {
        "Class": "Accuracy",
        "Precision": "-",
        "Recall": "-",
        "F1": round(float(accuracy), 3),
        "Support": int(support.sum()),
    }
)

metrics_df = pd.DataFrame(metrics_table)
display(metrics_df)
print(f"Validation accuracy: {accuracy:.3f}")


In [ ]:
# Confusion matrix visualization for validation set predictions
cm = confusion_matrix(val_y_classes, val_pred_classes, labels=labels)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([label_names[label] for label in labels])
ax.set_yticklabels([label_names[label] for label in labels])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Validation Confusion Matrix")
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, int(value), ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
coeffs = model.coef_[:TOTAL_PIECE_FEATURES]
print(f"Coefficient vector shape: {coeffs.shape}")

# Reshape into (piece, square)
modifier_matrix = coeffs.reshape(len(PIECES), FEATURES_PER_PIECE)

def normalize_piece_row(values: np.ndarray) -> np.ndarray:
    centered = values - values.mean()
    clipped = np.clip(centered, -40.0, 40.0)
    return clipped

json_payload: dict[str, dict[str, float]] = {}
for piece_idx, piece in enumerate(PIECES):
        normalized = normalize_piece_row(modifier_matrix[piece_idx])
        json_payload[piece] = {
            square: round(float(value), 3)
            for square, value in zip(SQUARES, normalized)
        }

print(f"Prepared modifiers for {len(json_payload)} piece types")

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as handle:
    json.dump(json_payload, handle, indent=2, sort_keys=True)
print(f"Wrote modifiers to {OUTPUT_JSON}")

In [ ]:
# Optional: regenerate Chess/LocationModifer.pm directly from the notebook.
run_perl_update = False  # set True to apply changes
if run_perl_update:
    result = subprocess.run(
        ["perl", str(UPDATE_SCRIPT), str(OUTPUT_JSON)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Failed to update LocationModifer.pm")

## Next Steps
- Re-run `perl perft.pl 4` and a self-play suite to validate the new tables.
- Commit `data/location_modifiers.json` and the regenerated `Chess/LocationModifer.pm` for review.